In [4]:
import numpy as np # linear algebra
import pandas as pd 
pd.set_option('max_columns', 100)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [6]:
print("Input Data Files :\n")
!ls input_data
print("\n\nOutput Data Files :\n")
!ls output_data

Input Data Files :

SampleSubmission.csv  test_customers.csv  train_customers.csv  vendors.csv
orders.csv	      test_locations.csv  train_locations.csv


Output Data Files :



In [7]:
INPUT_PATH = './input_data/'
OUTPUT_PATH = './output_data/'

train_customers = pd.read_csv(INPUT_PATH + 'train_customers.csv')
test_customers = pd.read_csv(INPUT_PATH + 'test_customers.csv')
test_locations = pd.read_csv(INPUT_PATH + 'test_locations.csv')
sample_sub = pd.read_csv(INPUT_PATH + 'SampleSubmission.csv')
orders = pd.read_csv(INPUT_PATH + 'orders.csv')
vendors = pd.read_csv(INPUT_PATH + 'vendors.csv')
train_locations = pd.read_csv(INPUT_PATH + 'train_locations.csv')

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (15,16,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [8]:
### Customers Data Cleaning and preprocessing

### Removing duplicate customer_ids
test_customers = test_customers.iloc[test_customers['akeed_customer_id'].drop_duplicates().index].reset_index(drop=True)
train_customers = train_customers.iloc[train_customers['akeed_customer_id'].drop_duplicates().index].reset_index(drop=True)

df_customers = pd.concat([train_customers, test_customers], axis=0)
df_customers['created_at'] = pd.to_datetime(df_customers['created_at'], format='%Y-%m-%d %H:%M:%S')
df_customers['updated_at'] = pd.to_datetime(df_customers['updated_at'], format='%Y-%m-%d %H:%M:%S')

for dcol in ['created_at', 'updated_at']:
    for attr in ['year', 'month']:
        df_customers[dcol + '_' + attr] = getattr(df_customers[dcol].dt, attr)
df_customers['updation_creation_diff'] = (df_customers['updated_at'] - df_customers['created_at']).dt.days
df_customers[['language', 'gender']] = df_customers[['language', 'gender']].apply(lambda x: pd.factorize(x)[0])
df_customers = df_customers.drop(['dob', 'created_at', 'updated_at', 'language'], axis=1).fillna(-1)
df_customers = df_customers.rename({'akeed_customer_id': 'customer_id'}, axis=1)

train_customers, test_customers = df_customers[:train_customers.shape[0]].reset_index(drop=True), df_customers[train_customers.shape[0]:].reset_index(drop=True)

print(train_customers.shape, test_customers.shape)
train_customers =  train_customers.drop_duplicates()
test_customers = test_customers.drop_duplicates()
train_customers.shape, test_customers.shape

(34523, 9) (9753, 9)


((34523, 9), (9753, 9))

In [9]:
### Cleaning and Preprocessing location information

train_locations = train_locations.drop_duplicates()
test_locations = test_locations.drop_duplicates()
test_locations.shape, train_locations.shape

##### almost half of the values are NaN
print('NaN values % is', end = '')
train_locations['location_type'].isnull().sum()/train_locations.shape[0], test_locations['location_type'].isnull().sum()/test_locations.shape[0]

### Filling NaN values with Unknown token
train_locations['location_type'] = train_locations['location_type'].fillna('UNK') # Unknown
test_locations['location_type'] = test_locations['location_type'].fillna('UNK')

### New values
print('NaN values % is', end = ' ', )
train_locations['location_type'].isnull().sum()/train_locations.shape[0], test_locations['location_type'].isnull().sum()/test_locations.shape[0]

((16720, 5), (59503, 5))

NaN values % is

(0.45727106196326234, 0.45753588516746413)

NaN values % is 

(0.0, 0.0)

In [10]:
### Merging Locations and Customers table

train_locations = pd.merge(train_locations, train_customers, on = 'customer_id', how = 'left')
test_locations = pd.merge(test_locations, test_customers, on = 'customer_id', how = 'left')
train_locations.shape, test_locations.shape

### Renaming to differentiate with vendor columns of the same name
train_locations = train_locations.rename({'latitude': 'cust_latitude', 'longitude': 'cust_longitude'}, axis=1)
test_locations = test_locations.rename({'latitude': 'cust_latitude', 'longitude': 'cust_longitude'}, axis=1)

### Ensuring there are no duplicate
test_locations = test_locations.drop_duplicates()
train_locatioons = train_locations.drop_duplicates()
train_locations.shape, test_locations.shape

((59503, 13), (16720, 13))

((59503, 13), (16720, 13))

In [11]:
### Vendors data cleaning and preprocessing

vendors['created_at'] = pd.to_datetime(vendors['created_at'], format='%Y-%m-%d %H:%M:%S')
vendors['updated_at'] = pd.to_datetime(vendors['updated_at'], format='%Y-%m-%d %H:%M:%S')
for dcol in ['created_at', 'updated_at']:
    for attr in ['year', 'month']:
        vendors['vendor_' + dcol + '_' + attr] = getattr(vendors[dcol].dt, attr)
vendors['updation_creation_diff'] = (vendors['updated_at'] - vendors['created_at']).dt.days
vendors = vendors.rename({'latitude': 'vendor_latitude', 'longitude': 'vendor_longitude'}, axis=1)


cat_cols = ['is_akeed_delivering', 'vendor_category_en']
vendors[cat_cols] = vendors[cat_cols].apply(lambda x: pd.factorize(x)[0])


### Vendors feature engineering

time_cols = ['sunday_from_time1', 'sunday_to_time1',
       'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1',
       'monday_to_time1', 'monday_from_time2', 'monday_to_time2',
       'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2',
       'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1',
       'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1',
       'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2',
       'friday_from_time1', 'friday_to_time1', 'friday_from_time2',
       'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1',
       'saturday_from_time2', 'saturday_to_time2']
vendors[time_cols] = vendors[time_cols].apply(lambda x: pd.to_datetime(x, errors='coerce'))
time_cols_1 = [c for c in time_cols if '1' in c]
time_cols_2 = [c for c in time_cols if '1' in c]
print(time_cols_1)

for wday in ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']:
    wday_cols = [c for c in time_cols_1 if wday in c]
    from_col, to_col = wday_cols[0], wday_cols[1]
    vendors[wday + 'hour_start'] = vendors[from_col].dt.hour 
    vendors[wday + 'hour_end'] = vendors[to_col].dt.hour
    vendors[wday + 'hour_diff'] = vendors[wday + 'hour_end'] - vendors[wday + 'hour_start']
    

### Time 2 cols
# for wday in ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']:
#     wday_cols = [c for c in time_cols_2 if wday in c]
#     from_col, to_col = wday_cols[0], wday_cols[1]
#     vendors[wday + 'hour_start_2'] = vendors[from_col].dt.hour 
#     vendors[wday + 'hour_end_2'] = vendors[to_col].dt.hour
#     vendors[wday + 'hour_diff_2'] = vendors[wday + 'hour_end_2'] - vendors[wday + 'hour_start_2']

vendors.head()

['sunday_from_time1', 'sunday_to_time1', 'monday_from_time1', 'monday_to_time1', 'tuesday_from_time1', 'tuesday_to_time1', 'wednesday_from_time1', 'wednesday_to_time1', 'thursday_from_time1', 'thursday_to_time1', 'friday_from_time1', 'friday_to_time1', 'saturday_from_time1', 'saturday_to_time1']


,id,authentication_id,vendor_latitude,vendor_longitude,vendor_category_en,vendor_category_id,delivery_charge,serving_distance,is_open,OpeningTime,OpeningTime2,prepration_time,commission,is_akeed_delivering,discount_percentage,status,verified,rank,language,vendor_rating,sunday_from_time1,sunday_to_time1,sunday_from_time2,sunday_to_time2,monday_from_time1,monday_to_time1,monday_from_time2,monday_to_time2,tuesday_from_time1,tuesday_to_time1,tuesday_from_time2,tuesday_to_time2,wednesday_from_time1,wednesday_to_time1,wednesday_from_time2,wednesday_to_time2,thursday_from_time1,thursday_to_time1,thursday_from_time2,thursday_to_time2,friday_from_time1,friday_to_time1,friday_from_time2,friday_to_time2,saturday_from_time1,saturday_to_time1,saturday_from_time2,saturday_to_time2,primary_tags,open_close_flags,vendor_tag,vendor_tag_name,one_click_vendor,country_id,city_id,created_at,updated_at,device_type,display_orders,vendor_created_at_year,vendor_created_at_month,vendor_updated_at_year,vendor_updated_at_month,updation_creation_diff,sundayhour_start,sundayhour_end,sundayhour_diff,mondayhour_start,mondayhour_end,mondayhour_diff,tuesdayhour_start,tuesdayhour_end,tuesdayhour_diff,wednesdayhour_start,wednesdayhour_end,wednesdayhour_diff,thursdayhour_start,thursdayhour_end,thursdayhour_diff,fridayhour_start,fridayhour_end,fridayhour_diff,saturdayhour_start,saturdayhour_end,saturdayhour_diff
0,4,118597.0,-0.588596,0.754434,0,2.0,0.0,6.0,1.0,11:00AM-11:30PM,-,15,0.0,0,0.0,1.0,1,11,EN,4.4,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 10:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 00:30:00,2020-05-20 10:00:00,2020-05-20 23:59:00,"{""primary_tags"":""4""}",1.0,"2,4,5,8,91,22,12,24,16,23","Arabic,Breakfast,Burgers,Desserts,Free Deliver...",Y,1.0,1.0,2018-01-30 14:42:04,2020-04-07 15:12:43,3,1,2018,1,2020,4,798,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13,118608.0,-0.471654,0.744470,0,2.0,0.7,5.0,1.0,08:30AM-10:30PM,-,14,0.0,0,0.0,1.0,1,11,EN,4.7,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 19:30:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 19:30:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,2020-05-20 00:00:00,2020-05-20 01:30:00,2020-05-20 08:00:00,2020-05-20 23:59:00,"{""primary_tags"":""7""}",1.0,"4,41,51,34,27,15,24,16,28","Breakfast,Cakes,Crepes,Italian,Pasta,Pizzas,Sa...",Y,1.0,1.0,2018-05-03 12:32:06,2020-04-05 20:46:03,3,1,2018,5,2020,4,703,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0
2,20,118616.0,-0.407527,0.643681,0,2.0,0.0,8.0,1.0,08:00AM-10:45PM,-,19,0.0,0,0.0,1.0,1,1,EN,4.5,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,2020-05-20 08:00:00,2020-05-20 22:45:00,NaT,NaT,"{""primary_tags"":""71""}",1.0,"4,8,91,10","Breakfast,Desserts,Free Delivery,Indian",Y,1.0,1.0,2018-05-04 22:28:22,2020-04-07 16:35:55,3,1,2018,5,2020,4,703,8.0,22.0,14.0,8,22,14,8.0,22.0,14.0,8,22,14,8.0,22.0,14.0,8.0,22.0,14.0,8.0,22.0,14.0
3,23,118619.0,-0.585385,0.753811,0,2.0,0.0,5.0,1.0,10:59AM-10:30PM,-,16,0.0,0,0.0,1.0,1,11,EN,4.5,202

In [12]:
### Removing datetime and object columns
''' Later datetime columns can be used to add more features '''

to_drop_cols = vendors.select_dtypes(['object', 'datetime64[ns]']).columns.tolist()
print(to_drop_cols)
vendors = vendors.drop(to_drop_cols + ['authentication_id'], axis=1)
vendors = vendors.rename({'id': 'vendor_id'}, axis=1)

' Later datetime columns can be used to add more features '

['OpeningTime', 'OpeningTime2', 'language', 'sunday_from_time1', 'sunday_to_time1', 'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1', 'monday_to_time1', 'monday_from_time2', 'monday_to_time2', 'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2', 'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1', 'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1', 'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2', 'friday_from_time1', 'friday_to_time1', 'friday_from_time2', 'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1', 'saturday_from_time2', 'saturday_to_time2', 'primary_tags', 'vendor_tag', 'vendor_tag_name', 'one_click_vendor', 'created_at', 'updated_at']


In [13]:
cust_ids_train = pd.Series(train_locations.customer_id.unique()).sample(frac = 1, random_state=2)
print(len(cust_ids_train))
train_locations = train_locations[train_locations.customer_id.isin(cust_ids_train)]
print(train_locations.shape)

35400
(59503, 13)


In [14]:
train_locations['key'] = 0
vendors['key'] = 0
train = pd.merge(train_locations, vendors, on = 'key')

In [15]:
test_locations['key'] = 0
test = pd.merge(test_locations, vendors, on = 'key')

In [16]:
icol = 'CID X LOC_NUM X VENDOR'
train['CID X LOC_NUM X VENDOR'] = train['customer_id'].astype('str') + ' X ' + train['location_number'].astype('str') + ' X ' + train['vendor_id'].astype('str')
test['CID X LOC_NUM X VENDOR'] = test['customer_id'].astype('str') + ' X ' + test['location_number'].astype('str') + ' X ' + test['vendor_id'].astype('str')

In [17]:
### Highly imbalanced dataset
print(train[icol].isin(orders[icol]).sum(), test[icol].isin(orders[icol]).sum())

80142 0


In [18]:
train['target'] =  train[icol].isin(orders[icol])
test['target'] = test[icol].isin(orders[icol])

In [19]:
train['target'] *= 1

In [20]:
train.to_pickle(OUTPUT_PATH + 'train.pkl')
test.to_pickle(OUTPUT_PATH  + 'test.pkl')

In [21]:
train.shape, test.shape

((5950300, 63), (1672000, 63))

In [22]:
print("Input Data Files :\n")
!ls input_data
print("\n\nOutput Data Files :\n")
!ls output_data

Input Data Files :

SampleSubmission.csv  test_customers.csv  train_customers.csv  vendors.csv
orders.csv	      test_locations.csv  train_locations.csv


Output Data Files :

test.pkl  train.pkl
